# Dataset Understanding — Stage 1: Structural Understanding

This notebook performs Stage 1 (Structural Understanding) of the Dataset Understanding Strategy: row counts, primary key uniqueness, referential
integrity between tables, join feasibility, and quarter consistency.

**Scope:** structural checks only. No feature engineering, no distributions, visualizations, or correlations, and no SQL/database work — those belong to later steps. Raw CSVs are loaded directly with pandas here for inspection only; this does not duplicate or replace `ingest.py`'s verification logic.

In [ ]:
import pandas as pd

RAW_DIR = "../data/raw"

demographics = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_demographics.csv")
location = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_location.csv")
population = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_population.csv")
services = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_services.csv")
status = pd.read_csv(f"{RAW_DIR}/telco_customer_churn_status.csv")

tables = {
    "Demographics": demographics,
    "Location": location,
    "Population": population,
    "Services": services,
    "Status": status,
}

## 1. Row Counts

In [ ]:
EXPECTED_ROWS = {
    "Demographics": 7043,
    "Location": 7043,
    "Population": 1671,
    "Services": 7043,
    "Status": 7043,
}

for name, df in tables.items():
    n_rows, n_cols = df.shape
    expected = EXPECTED_ROWS[name]
    status_label = "PASS" if n_rows == expected else "FAIL"
    print(f"[{status_label}] {name}: {n_rows} rows, {n_cols} columns (expected {expected} rows)")

## 2. Primary Key Uniqueness

In [ ]:
pk_columns = {
    "Demographics": "Customer ID",
    "Location": "Customer ID",
    "Services": "Customer ID",
    "Status": "Customer ID",
    "Population": "Zip Code",
}

for name, key in pk_columns.items():
    df = tables[name]
    n_duplicates = int(df[key].duplicated().sum())
    status_label = "PASS" if n_duplicates == 0 else "FAIL"
    print(f"[{status_label}] {name}: {n_duplicates} duplicate '{key}' values")

## 3. Referential Integrity (Customer ID)

In [ ]:
demographics_ids = set(demographics["Customer ID"])

for name in ["Location", "Services", "Status"]:
    df = tables[name]
    table_ids = set(df["Customer ID"])

    missing_from_demographics = table_ids - demographics_ids
    missing_from_table = demographics_ids - table_ids

    status_label = "PASS" if not missing_from_demographics and not missing_from_table else "FAIL"
    print(f"[{status_label}] {name} <-> Demographics:")
    print(f"    {name} IDs not found in Demographics: {len(missing_from_demographics)}")
    print(f"    Demographics IDs not found in {name}: {len(missing_from_table)}")

## 4. Join Feasibility (Zip Code)

In [ ]:
population_zips = set(population["Zip Code"])
location_zips = location["Zip Code"]

unmatched_mask = ~location_zips.isin(population_zips)
n_unmatched = int(unmatched_mask.sum())

status_label = "PASS" if n_unmatched == 0 else "FAIL"
print(f"[{status_label}] Location rows with a Zip Code not found in Population: {n_unmatched}")

## 5. Quarter Consistency

In [ ]:
for name in ["Services", "Status"]:
    df = tables[name]
    value_counts = df["Quarter"].value_counts()
    n_distinct = df["Quarter"].nunique()
    status_label = "PASS" if n_distinct == 1 else "FAIL"
    print(f"[{status_label}] {name}: {n_distinct} distinct Quarter value(s)")
    print(value_counts.to_string())
    print()

## 6. Summary

Stage 1 structural checks cover: 
- row counts for all five raw tables against the expected spec used by `ingest.py`;
- `Customer ID` uniqueness in Demographics, Location, Services, and Status;
- `Zip Code` uniqueness in Population;
- Customer ID referential integrity between Demographics and each of Location, Services, and Status;
- Zip Code join feasibility between Location and Population; 
- Quarter consistency in Services and Status.

If every check above printed `PASS`, the five raw tables are structurally sound for downstream joining: unique keys, no orphaned foreign keys, and a single consistent Quarter. Any `FAIL` above should be treated as a finding to investigate before relying on that relationship in later stages.